# OpenCV 二维码识别验证

只测试 OpenCV 是否能识别 A01 定位牌二维码。不调用大模型，不移动小车。建议手动把小车放在不同距离，多次运行记录结果。

In [ ]:
# 1. 配置

# 手动填写当前距离标签，例如 100cm / 80cm / 60cm / 50cm。
DISTANCE_LABEL = "unknown"

OUTPUT_DIR = "opencv_qr_validation_output_" + DISTANCE_LABEL

# 云台正前方标定值。
HOME_PAN = 2048
HOME_TILT = 2868

# 先用 640x480；如果相机不稳定，再改 224x224。
CAMERA_WIDTH = 640
CAMERA_HEIGHT = 480

SAMPLE_COUNT = 12
INTERVAL_SECONDS = 0.20


In [ ]:
# 2. 运行验证
from opencv_qr_validation import run_demo, summarize_report

report = run_demo(
    output_dir=OUTPUT_DIR,
    distance_label=DISTANCE_LABEL,
    init_gimbal=True,
    home_pan=HOME_PAN,
    home_tilt=HOME_TILT,
    camera_width=CAMERA_WIDTH,
    camera_height=CAMERA_HEIGHT,
    sample_count=SAMPLE_COUNT,
    interval_seconds=INTERVAL_SECONDS,
)

print(summarize_report(report))


In [ ]:
# 3. 显示最佳标注图
from IPython.display import Image, display

best = None
for sample in report.get("samples", []):
    if sample.get("target_ok"):
        best = sample
        break
if best is None and report.get("samples"):
    best = report["samples"][0]

if best:
    print("sample", best.get("index"), best.get("annotated_image_path"))
    display(Image(filename=best.get("annotated_image_path")))


In [ ]:
# 4. 查看每张样本简表
import json

for sample in report.get("samples", []):
    det = sample.get("detection") or {}
    print(json.dumps({
        "index": sample.get("index"),
        "ok": sample.get("ok"),
        "target_ok": sample.get("target_ok"),
        "method": det.get("method"),
        "qr_width_px": det.get("qr_width_px"),
        "payload": det.get("payload"),
    }, ensure_ascii=False))
